# Lab Day 2 — Backbone, công thức huấn luyện và suy luận trên DeepWeeds

Notebook **khung** cho Colab hoặc Kaggle. Các ô có chữ `TODO` là phần **bạn tự viết**; ô cài đặt, tải dữ liệu
và chạy `eval.py` đã viết sẵn. Đọc `README.md` (quy tắc chia dữ liệu, cách đánh giá), `GUIDE.md` và `RUBRIC.md` trước.

**Quy tắc quan trọng nhất:** chọn mọi thứ trên **val**; **test chỉ chạy một lần mỗi seed ở Bước 4**.

Bản đồ: Bước 0 (EDA, kiểm tra) → Bước 1 (backbone) → Bước 2 (công thức huấn luyện) → Bước 3 (suy luận)
→ Bước 4 (chung kết + `eval.py`) → Bước 5 (xlsx, biểu đồ, báo cáo).

**Chạy trên Kaggle:** *File → Import Notebook* (tải file này lên) → *Settings*: **Accelerator = GPU T4 x2**,
**Internet = On** → *Run All* (chạy dài thì dùng *Save Version → Save & Run All*).
Code được `git clone` từ GitHub nên **phải `git push` trước** mỗi lần chạy. Mọi kết quả (`runs/`, `predictions/`,
`curves/`, `figures/`, `env.json`) nằm ở `/kaggle/working` → tab *Output*; dữ liệu nằm ở `/kaggle/temp` (không bị lưu).

## 0. Cài đặt môi trường

In [ ]:
# Cài thư viện (Colab/Kaggle). Kaggle: Settings → Accelerator = GPU T4 x2, Internet = On.
%pip install -q timm openpyxl

import sys, os, json, platform, subprocess
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").is_dir()
ON_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

REPO_URL = "https://github.com/AIVIETNAM-AIO-AnhDinh/K4-Track4-Day2-DinhLenhTienAnh-2A202602928.git"
BRANCH = "main"
SUBMISSION = "submissions/2A202602928_DinhLenhTienAnh"   # thư mục bài nộp trong repo

if ON_KAGGLE or ON_COLAB:
    # Kaggle: /kaggle/working được lưu ở tab Output; /kaggle/temp thì không (dataset, repo để ở đây).
    WORK, SCRATCH = (Path("/kaggle/working"), Path("/kaggle/temp")) if ON_KAGGLE else (Path("/content"), Path("/content"))
    SCRATCH.mkdir(parents=True, exist_ok=True)
    REPO_DIR = SCRATCH / "repo"
    if (REPO_DIR / ".git").exists():
        !git -C {REPO_DIR} pull -q
    else:
        !git clone -q --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
    OUT_DIR, DATA_DIR = WORK, SCRATCH / "data"
else:
    # Chạy local: notebook nằm trong <repo>/submissions/<...>/code
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "eval.py").exists())
    OUT_DIR, DATA_DIR = REPO_DIR / SUBMISSION, REPO_DIR / "data"

CODE_DIR = REPO_DIR / SUBMISSION / "code"
assert (REPO_DIR / "eval.py").exists() and (CODE_DIR / "dataset.py").exists(), \
    f"Không thấy eval.py hoặc {CODE_DIR}: đã git push code lên GitHub chưa?"
sys.path.insert(0, str(REPO_DIR))   # để import eval.py
sys.path.insert(0, str(CODE_DIR))   # dataset.py, model.py, train.py ... của bài nộp

RUNS_DIR, PRED_DIR, CURVES_DIR, FIG_DIR = (OUT_DIR / d for d in ("runs", "predictions", "curves", "figures"))
for d in (RUNS_DIR, PRED_DIR, CURVES_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

import numpy as np, pandas as pd, torch, torchvision, timm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = min(4, os.cpu_count() or 1)
SMOKE = os.environ.get("LAB_SMOKE") == "1"   # chỉ để thử code nhanh trên CPU (ít ảnh, 1 epoch): KHÔNG phải thí nghiệm thật
if DEVICE.type == "cuda":
    try:  # P100 (sm_60) không còn được các bản torch mới hỗ trợ: báo sớm thay vì lỗi giữa chừng
        (torch.ones(2, device=DEVICE) * 2).sum().item()
    except RuntimeError as e:
        raise RuntimeError(f"GPU {torch.cuda.get_device_name(0)} không chạy được với torch {torch.__version__}. "
                           "Kaggle: Settings → Accelerator → GPU T4 x2.") from e

git_rev = subprocess.run(["git", "-C", str(REPO_DIR), "rev-parse", "--short", "HEAD"],
                         capture_output=True, text=True).stdout.strip()
ENV = {
    "platform": "kaggle" if ON_KAGGLE else "colab" if ON_COLAB else platform.platform(),
    "python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
    "timm": timm.__version__, "numpy": np.__version__, "pandas": pd.__version__,
    "cuda": torch.version.cuda, "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else None,
    "cpu_count": os.cpu_count(), "git_commit": git_rev,
}
(OUT_DIR / "env.json").write_text(json.dumps(ENV, indent=2))   # phiên bản thư viện cho README/báo cáo
print(json.dumps(ENV, indent=2))
print("REPO_DIR =", REPO_DIR, "| OUT_DIR =", OUT_DIR, "| DATA_DIR =", DATA_DIR)
if DEVICE.type != "cuda":
    print("CẢNH BÁO: KHÔNG CÓ GPU. Kaggle: Settings → Accelerator → GPU T4 x2.")

### Tải dữ liệu (đã viết sẵn)
Ảnh từ Zenodo (~490 MB, kiểm tra MD5), nhãn và các file chia fold từ GitHub của tác giả.
Nếu đã có ảnh (ở local trong `<repo>/images`, hoặc một dataset DeepWeeds gắn vào *Kaggle Input*) thì bỏ qua bước tải.
`IMAGES_DIR` được tìm tự động theo một tên ảnh có thật trong DeepWeeds, không phụ thuộc cấu trúc thư mục trong zip.

In [ ]:
import hashlib, urllib.request, zipfile

ZIP_URL = "https://zenodo.org/records/7939060/files/images.zip?download=1"
ZIP_MD5 = "b7b30f96d466fba86016aa5a26606e0f"
LABELS_URL = "https://raw.githubusercontent.com/AlexOlsen/DeepWeeds/master/labels"
MARKER = "20160928-140314-0.jpg"   # một ảnh của DeepWeeds, dùng để tìm thư mục ảnh


def find_images_dir(*roots):
    for root in map(Path, roots):
        if root.is_dir():
            hit = next(root.rglob(MARKER), None)
            if hit is not None:
                return hit.parent
    return None


DATA_DIR.mkdir(parents=True, exist_ok=True)
search = [DATA_DIR] + ([Path("/kaggle/input")] if ON_KAGGLE else []) \
         + ([REPO_DIR / "images"] if not (ON_KAGGLE or ON_COLAB) else [])
IMAGES_DIR = find_images_dir(*search)
if IMAGES_DIR is None:
    zip_path = DATA_DIR / "images.zip"
    if not zip_path.exists():
        print("Đang tải images.zip (~490 MB) từ Zenodo ...")
        urllib.request.urlretrieve(ZIP_URL, zip_path)
    h = hashlib.md5()
    with open(zip_path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == ZIP_MD5, f"MD5 sai: {h.hexdigest()} (xoá {zip_path} rồi chạy lại)"
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(DATA_DIR / "images")
    IMAGES_DIR = find_images_dir(DATA_DIR)

LABELS_DIR = DATA_DIR / "labels"
LABELS_DIR.mkdir(exist_ok=True)
for name in ["labels", "train_subset0", "val_subset0", "test_subset0"]:
    dst = LABELS_DIR / f"{name}.csv"
    if not dst.exists():
        urllib.request.urlretrieve(f"{LABELS_URL}/{name}.csv", dst)

n_jpg = sum(1 for _ in IMAGES_DIR.glob("*.jpg"))
print("IMAGES_DIR =", IMAGES_DIR, "|", n_jpg, "ảnh .jpg")
print("LABELS_DIR =", LABELS_DIR, "|", sorted(p.name for p in LABELS_DIR.glob("*.csv")))
assert n_jpg == 17509, f"Kỳ vọng 17.509 ảnh, thấy {n_jpg}"

In [ ]:
# Đường dẫn dùng chung cho mọi thí nghiệm: run(Config(**PATHS, exp_id="B01", backbone="resnet50", seed=0))
PATHS = dict(images_dir=str(IMAGES_DIR), labels_dir=str(LABELS_DIR), out_dir=str(RUNS_DIR),
             pred_dir=str(PRED_DIR), curves_dir=str(CURVES_DIR), num_workers=NUM_WORKERS)
PATHS

## Bước 0 — EDA và kiểm tra pipeline
Mục tiêu: hiểu dữ liệu, chạy các kiểm tra chia dữ liệu (README mục 2.1), kiểm tra pipeline trước khi chạy thật.
Hình và số liệu của bước này được lưu vào `figures/` để đưa vào báo cáo.

### 0.1 Kiểm tra chia dữ liệu và phân bố lớp (README mục 2.1)

In [ ]:
import matplotlib.pyplot as plt
import dataset as D

train_df, val_df, test_df = D.load_split(LABELS_DIR, fold=0)
split_stats = D.check_split(train_df, val_df, test_df, IMAGES_DIR, labels_csv=LABELS_DIR / "labels.csv")

per_class = split_stats["per_class"]
per_class["Δ vs paper"] = per_class["total"] - per_class["paper"]
per_class.to_csv(FIG_DIR / "eda_per_class.csv")
mismatch = split_stats["label_mismatch"]
(FIG_DIR / "split_checks.json").write_text(json.dumps(
    {k: v for k, v in split_stats.items() if k not in ("per_class", "label_mismatch")}
    | {"label_mismatch": mismatch.to_dict("records")}, indent=2, ensure_ascii=False))

c_all, c_tr = per_class["total"], per_class["train"]
print(f"\nMất cân bằng (cả dataset): {c_all.idxmax()} {c_all.max()} / {c_all.idxmin()} {c_all.min()} = "
      f"{c_all.max() / c_all.min():.2f}x | Negatives chiếm {c_all['Negatives'] / c_all.sum():.1%}")
print(f"Mất cân bằng (train): {c_tr.max()} / {c_tr.min()} = {c_tr.max() / c_tr.min():.2f}x")
print("Lệch so với Table 1 của bài báo:", per_class.loc[per_class["Δ vs paper"] != 0, "Δ vs paper"].to_dict())

In [ ]:
share = per_class[["train", "val", "test"]] / per_class[["train", "val", "test"]].sum()
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
per_class[["train", "val", "test"]].plot.bar(ax=axes[0], width=0.8)
axes[0].set(title="DeepWeeds fold 0: số ảnh mỗi lớp", ylabel="số ảnh", xlabel="")
for cont in axes[0].containers:
    axes[0].bar_label(cont, fontsize=7, rotation=90, padding=2)
axes[0].set_ylim(0, per_class["train"].max() * 1.15)
(share * 100).plot.bar(ax=axes[1], width=0.8)
axes[1].set(title="Tỉ lệ lớp trong từng tập (%): giống nhau giữa train/val/test", ylabel="% số ảnh của tập", xlabel="")
for ax in axes:
    ax.tick_params(axis="x", rotation=35)
    plt.setp(ax.get_xticklabels(), ha="right")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_class_distribution.png", dpi=150)
plt.show()

### 0.2 Xem ảnh mẫu (3 ảnh mỗi lớp, tập train)

In [ ]:
from PIL import Image

samples = train_df.groupby("Label").sample(n=3, random_state=0).sort_values("Label")
fig, axes = plt.subplots(3, D.NUM_CLASSES, figsize=(2.1 * D.NUM_CLASSES, 7))
for col, (label, grp) in enumerate(samples.groupby("Label")):
    for row, fname in enumerate(grp["Filename"]):
        ax = axes[row, col]
        ax.imshow(Image.open(IMAGES_DIR / fname))
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_xlabel(fname, fontsize=6)
        if row == 0:
            ax.set_title(f"{label}: {D.CLASS_NAMES[label]}", fontsize=9)
fig.suptitle("DeepWeeds: ảnh mẫu theo lớp (train, fold 0)")
fig.tight_layout()
fig.savefig(FIG_DIR / "eda_samples.jpg", dpi=110, pil_kwargs={"quality": 85})
plt.show()

### 0.3 Thống kê ảnh (kích thước, kênh, mean/std) để chọn chuẩn hoá

In [ ]:
from collections import Counter

all_files = pd.concat([train_df, val_df, test_df])["Filename"]
size_mode = Counter()
for f in all_files:
    with Image.open(IMAGES_DIR / f) as im:   # chỉ đọc header, nhanh
        size_mode[(im.size, im.mode)] += 1
print("(kích thước, mode) của toàn bộ ảnh:", dict(size_mode))

# mean/std theo kênh trên 1.000 ảnh train ngẫu nhiên (chỉ train, quy tắc S4)
sub = train_df.sample(n=1000, random_state=0)["Filename"]
px = np.stack([np.asarray(Image.open(IMAGES_DIR / f).convert("RGB"), dtype=np.float32) / 255 for f in sub])
ch_mean, ch_std = px.mean(axis=(0, 1, 2)), px.std(axis=(0, 1, 2))
print("DeepWeeds train mean:", ch_mean.round(3), "std:", ch_std.round(3))
print("ImageNet       mean:", np.round(D.IMAGENET_MEAN, 3), "std:", np.round(D.IMAGENET_STD, 3))
print("→ Dùng mean/std của trọng số tiền huấn luyện (model.data_config), vì backbone học trên phân phối đó.")
del px

### 0.4 Kiểm tra pipeline (slide trang 59, GUIDE.md mục 1.3)
1. cố định seed · 2. loss ban đầu ≈ −ln(1/9) = 2,197 · 3. overfit một batch nhỏ tới loss ≈ 0
· 4. ảnh sau augmentation (giải chuẩn hoá) khớp nhãn · 5. `model.train()` / `model.eval()` đúng lúc.

**Loss ban đầu.** Chạy cho mọi backbone gợi ý (cũng kiểm tra luôn tên model/tag tải được trước Bước 1).
Cột `loss_timm_init` dùng head khởi tạo mặc định của timm, cột `loss` dùng head khởi tạo lại bằng
`model.reset_head` (mặc định của `build_model`, áp dụng giống nhau cho mọi backbone).

In [ ]:
import torch.nn.functional as F
from train import set_seed
from model import (SUGGESTED_BACKBONES, build_model, reset_head, weight_tag, data_config,
                   param_groups, set_train_mode, count_params, count_gmacs)

SEED = 0
LN9 = float(np.log(D.NUM_CLASSES))


def mean_loss(model, loader, n_batches=2):
    model.eval()
    out = []
    with torch.inference_mode():
        for i, (x, y, _) in enumerate(loader):
            if i == n_batches:
                break
            out.append(F.cross_entropy(model(x.to(DEVICE)), y.to(DEVICE)).item())
    return float(np.mean(out))


rows = []
for name in SUGGESTED_BACKBONES.values():
    set_seed(SEED)
    m = build_model(name, pretrained=True, head_std=None).to(DEVICE)   # head mặc định của timm
    cfg_m = data_config(m)
    loader = D.make_loader(val_df, IMAGES_DIR, D.build_transforms(False, 224, mean=cfg_m["mean"], std=cfg_m["std"]),
                           batch_size=64, train=False, num_workers=NUM_WORKERS)
    loss_timm = mean_loss(m, loader)
    reset_head(m)                                                         # như build_model(...) mặc định
    rows.append({"backbone": name, "weight_tag": weight_tag(m), "params_M": round(count_params(m), 2),
                 "GMAC@224": round(count_gmacs(m, 224), 3), "mean": cfg_m["mean"], "std": cfg_m["std"],
                 "loss_timm_init": round(loss_timm, 4), "loss": round(mean_loss(m, loader), 4)})
    del m, loader
    torch.cuda.empty_cache()

init_df = pd.DataFrame(rows)
init_df["|loss - ln9|"] = (init_df["loss"] - LN9).abs().round(4)
init_df.to_csv(FIG_DIR / "check_initial_loss.csv", index=False)
print(f"Kỳ vọng -ln(1/9) = {LN9:.4f} (2 batch val, eval mode)")
display(init_df)
assert (init_df["|loss - ln9|"] < 0.1).all(), "Loss ban đầu lệch xa ln 9: kiểm tra head/nhãn/chuẩn hoá"

**Overfit một batch nhỏ** (2 ảnh mỗi lớp = 18 ảnh, không augmentation) với backbone mốc. Không xuống được gần 0
thì lỗi nằm ở code/model, chưa chạy tiếp.

In [ ]:
CHECK_BACKBONE = "resnet18" if SMOKE else "resnet50"   # backbone mốc; đổi để kiểm tra backbone khác
STEPS = 150

set_seed(SEED)
model = build_model(CHECK_BACKBONE, pretrained=True).to(DEVICE)
cfg_m = data_config(model)
MEAN, STD = cfg_m["mean"], cfg_m["std"]
val_tf = D.build_transforms(False, 224, mean=MEAN, std=STD)

tiny = train_df.groupby("Label").sample(n=2, random_state=SEED)
xb, yb, _ = next(iter(D.make_loader(tiny, IMAGES_DIR, val_tf, batch_size=len(tiny), train=False, num_workers=0)))
xb, yb = xb.to(DEVICE), yb.to(DEVICE)

opt = torch.optim.AdamW(param_groups(model, lr_backbone=1e-4, lr_head=1e-3, weight_decay=0.0))
use_amp = DEVICE.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
overfit_hist = []
for step in range(STEPS):
    set_train_mode(model)
    with torch.autocast(DEVICE.type, dtype=torch.float16, enabled=use_amp):
        loss = F.cross_entropy(model(xb), yb)
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.step(opt)
    scaler.update()
    overfit_hist.append(loss.item())
    if step % 25 == 0 or step == STEPS - 1:
        print(f"step {step:3d}  loss {loss.item():.4f}")

model.eval()
with torch.inference_mode():
    overfit_acc = (model(xb).argmax(1) == yb).float().mean().item()
print(f"Sau {STEPS} bước: loss train {overfit_hist[-1]:.4f} | accuracy eval-mode trên batch đó {overfit_acc:.0%}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.semilogy(overfit_hist)
ax.axhline(LN9, ls="--", c="gray", label="ln 9")
ax.set(title=f"Overfit 1 batch ({len(tiny)} ảnh) - {CHECK_BACKBONE}", xlabel="bước", ylabel="loss CE (log)")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "check_overfit_one_batch.png", dpi=150)
plt.show()
assert overfit_hist[-1] < 0.05 and overfit_acc == 1.0, "Không overfit được 1 batch: kiểm tra code trước khi chạy tiếp"

**Ảnh sau augmentation.** Hàng trên: batch thật lấy từ train loader (`aug="basic"`, đã giải chuẩn hoá) với nhãn do
loader trả về. Hàng dưới: ảnh gốc đọc lại từ đĩa theo tên file, với nhãn tra trong CSV. Hai hàng phải cùng cây, cùng nhãn.

In [ ]:
train_tf = D.build_transforms(True, 224, aug="basic", mean=MEAN, std=STD)
aug_loader = D.make_loader(train_df, IMAGES_DIR, train_tf, batch_size=8, train=True, num_workers=0, seed=SEED)
xa, ya, fa = next(iter(aug_loader))
csv_label = train_df.set_index("Filename")["Label"]
assert all(csv_label[f] == int(y) for f, y in zip(fa, ya)), "Nhãn của loader không khớp CSV"

fig, axes = plt.subplots(2, len(fa), figsize=(2.3 * len(fa), 5.9))
for j, (img, y, f) in enumerate(zip(D.denormalize(xa, MEAN, STD), ya, fa)):
    axes[0, j].imshow(img.permute(1, 2, 0).numpy())
    axes[0, j].set_title(f"loader: {D.CLASS_NAMES[int(y)]}", fontsize=8)
    axes[1, j].imshow(Image.open(IMAGES_DIR / f))
    axes[1, j].set_title(f"CSV: {D.CLASS_NAMES[csv_label[f]]}", fontsize=8)
    axes[1, j].set_xlabel(f, fontsize=6)
for ax in axes.flat:
    ax.set_xticks([]); ax.set_yticks([])
fig.suptitle("Train loader sau augmentation (trên) và ảnh gốc theo tên file (dưới)")
fig.tight_layout()
fig.savefig(FIG_DIR / "check_augmented_batch.jpg", dpi=110, pil_kwargs={"quality": 85})
plt.show()

# Các mức augmentation của trục B (dùng ở Bước 2) trên cùng một ảnh
img0 = Image.open(IMAGES_DIR / fa[0]).convert("RGB")
modes = [a for a in D.AUG_CHOICES if a != "none"]
fig, axes = plt.subplots(len(modes), 6, figsize=(12, 2.1 * len(modes)))
torch.manual_seed(SEED)
for i, mode in enumerate(modes):
    tf = D.build_transforms(True, 224, aug=mode, mean=MEAN, std=STD)
    for j in range(6):
        axes[i, j].imshow(D.denormalize(tf(img0), MEAN, STD).permute(1, 2, 0).numpy())
        axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
    axes[i, 0].set_ylabel(mode)
fig.suptitle(f"Các mức aug trên {fa[0]} ({D.CLASS_NAMES[int(ya[0])]})")
fig.tight_layout()
fig.savefig(FIG_DIR / "check_aug_modes.jpg", dpi=100, pil_kwargs={"quality": 85})
plt.show()

**`train()` / `eval()`.** Ở eval mode, BatchNorm dùng thống kê đã lưu nên dự đoán của một ảnh không phụ thuộc
các ảnh khác trong batch; ở train mode thì có. Vì vậy mọi lượt đánh giá phải `model.eval()` + `torch.inference_mode()`.

In [ ]:
import copy

probe = copy.deepcopy(model)   # bản sao: forward ở train mode sẽ cập nhật running stats của BN
x8 = xb[:8]
probe.eval()
with torch.inference_mode():
    diff_eval = (probe(x8[:1]) - probe(x8)[:1]).abs().max().item()
probe.train()
with torch.inference_mode():
    diff_train = (probe(x8[:1]) - probe(x8)[:1]).abs().max().item()
print(f"Lệch logit của cùng 1 ảnh khi chạy riêng vs trong batch 8: eval {diff_eval:.2e} | train {diff_train:.2e}")
assert diff_eval < 1e-3 < diff_train
del probe

pipeline_checks = {
    "seed": SEED, "ln9": LN9,
    "initial_loss": init_df.set_index("backbone")[["loss_timm_init", "loss"]].to_dict("index"),
    "overfit": {"backbone": CHECK_BACKBONE, "weight_tag": weight_tag(model), "n_images": len(tiny),
                "steps": STEPS, "final_loss": overfit_hist[-1], "eval_acc": overfit_acc},
    "bn_mode_logit_diff": {"eval": diff_eval, "train": diff_train},
}
(FIG_DIR / "pipeline_checks.json").write_text(json.dumps(pipeline_checks, indent=2))
del model, opt
torch.cuda.empty_cache()
print("Bước 0 xong. Hình và số liệu ở", FIG_DIR)

## Cấu hình chạy Bước 1-5
Mọi thí nghiệm đi qua `train.run(Config(...))`. Có ≥ 2 GPU (Kaggle T4 x2) thì `experiments.run_many` chạy song song
mỗi GPU một cấu hình (tiến trình `python train.py --set ...`, log ở `runs/<exp_id>/seed<k>/train.log`).
Đã có `summary.json` thì không chạy lại (chạy lại notebook trong cùng phiên sẽ tiếp tục từ chỗ dừng).

**Quy tắc quyết định đặt trước khi chạy (chỉ dùng VAL):**
- Bước 1: chọn backbone có macro-F1 val cao nhất; các backbone kém hơn không quá `NOISE` = 0,01 (1 seed, không phân
  biệt được) thì chọn cái có độ trễ batch-1 thấp nhất.
- Bước 2: mỗi trục, giá trị tốt nhất thắng nếu hơn `T00` quá `MIN_GAIN`; ≥ 2 trục thắng thì chạy thêm kết hợp `T12`.
  Cấu hình cuối = lần chạy T có macro-F1 val cao nhất.
- Bước 3: phương pháp suy luận cuối = macro-F1 val cao nhất trong các cách 1 mô hình có p95 batch-1 ≤ 100 ms;
  luôn thêm temperature scaling (T khớp trên val).

In [ ]:
import copy, shutil
from dataclasses import replace, asdict
import experiments as X
import inference as INF
import benchmark as BM
from train import Config, run_dir, curve_path, _subset
from eval import save_predictions, compute_metrics, read_pred

N_PARALLEL = int(os.environ.get("LAB_PARALLEL", max(1, torch.cuda.device_count())))
BASE = dict(PATHS, preload=True, num_workers=max(1, NUM_WORKERS // N_PARALLEL))
if SMOKE:
    BASE.update(epochs=1, batch_size=16, debug_subset=6)
LAT = dict(device=str(DEVICE), warmup=2 if SMOKE else 10, iters=5 if SMOKE else 100)
SYNC = torch.cuda.synchronize if DEVICE.type == "cuda" else None
NOISE = 0.01
LATENCY_BUDGET_MS = 100.0
VAL_DF, TEST_DF = _subset(val_df, BASE.get("debug_subset")), _subset(test_df, BASE.get("debug_subset"))
HARD = ["Chinee Apple", "Snake Weed"]


def f1_of(s, cls):
    return s["val_f1_per_class"][cls]


print(f"N_PARALLEL={N_PARALLEL} | num_workers/tiến trình={BASE['num_workers']} | SMOKE={SMOKE}")

## Bước 1 — So sánh backbone (≥ 5)
Cùng công thức nền `T00` (GUIDE.md mục 1.4), cùng seed, mỗi backbone một lần chạy. Mã thí nghiệm `B01`, `B02`, ...

In [ ]:
BACKBONES = {   # exp_id: (tên timm, mô tả)
    "B01": ("resnet50", "resnet50"),
    "B02": ("convnext_tiny", "convnext_tiny"),
    "B03": ("deit_small_patch16_224", "deit_small"),
    "B04": ("swin_tiny_patch4_window7_224", "swin_tiny"),
    "B05": ("efficientnet_b0", "efficientnet_b0"),
    "B06": ("mobilenetv3_large_100", "mobilenetv3_large"),
}
if SMOKE:
    BACKBONES = {"B01": ("resnet18", "resnet18"), "B02": ("convnext_atto", "convnext_atto"),
                 "B03": ("vit_tiny_patch16_224", "vit_tiny"), "B04": ("mobilenetv3_small_050", "mbv3_small")}

b_cfgs = {e: Config(**BASE, exp_id=e, backbone=n, desc=d, seed=0) for e, (n, d) in BACKBONES.items()}
b_sum = dict(zip(b_cfgs, X.run_many(list(b_cfgs.values()), N_PARALLEL)))

In [ ]:
# Độ trễ sơ bộ batch 1 (đo đúng cách: warmup, synchronize, 100 lần), khi không có tiến trình train nào chạy
lat_rows = []
for e, cfg in b_cfgs.items():
    m, _ = X.load_model(run_dir(cfg), DEVICE)
    for dt in ("fp32", "amp"):
        lat_rows.append({"config": f"{e} {cfg.desc}", "step": "B", **BM.latency_report(m, 1, 224, dt, **LAT)})
    del m
    torch.cuda.empty_cache()
lat_df = pd.DataFrame(lat_rows)
lat_b1 = lat_df[lat_df.dtype == "fp32"].set_index("config")["p50"]

bb = pd.DataFrame([{
    "exp_id": e, "backbone": s["backbone"], "weight_tag": s["weight_tag"], "params_M": s["params_M"],
    "GMAC": s["gmacs"], "img_size": s["img_size"], "epochs": s["epochs"], "seed": s["seed"],
    "val_macro_f1": s["val_macro_f1"], "val_top1": s["val_top1"], "val_balanced_acc": s["val_balanced_acc"],
    "best_epoch": s["best_epoch"], "train_s_per_epoch": s["train_time_per_epoch_s"],
    "latency_b1_fp32_p50_ms": lat_b1[f"{e} {b_cfgs[e].desc}"],
    "F1_ChineeApple": f1_of(s, "Chinee Apple"), "F1_SnakeWeed": f1_of(s, "Snake Weed"),
} for e, s in b_sum.items()])

best_f1 = bb.val_macro_f1.max()
cands = bb[bb.val_macro_f1 >= best_f1 - NOISE].sort_values("latency_b1_fp32_p50_ms")
chosen = cands.iloc[0]
BB, BB_EXP = chosen.backbone, chosen.exp_id
top = bb.loc[bb.val_macro_f1.idxmax()]
BB_REASON = (f"{BB_EXP} {BB}: macro-F1 val {chosen.val_macro_f1:.4f} (cao nhất {top.val_macro_f1:.4f} là {top.exp_id} "
             f"{top.backbone}); trong {len(cands)} backbone cách mức cao nhất ≤ {NOISE}, đây là cái có độ trễ batch-1 "
             f"thấp nhất ({chosen.latency_b1_fp32_p50_ms:.2f} ms FP32).")
bb["note"] = np.where(bb.exp_id == BB_EXP, "CHỌN đi tiếp", np.where(bb.val_macro_f1 >= best_f1 - NOISE, "trong nhiễu so với tốt nhất", ""))
display(bb.round(4))
print("Chọn:", BB_REASON)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(bb.latency_b1_fp32_p50_ms, bb.val_macro_f1, s=bb.params_M * 8, alpha=0.6)
for _, r in bb.iterrows():
    ax.annotate(f"{r.exp_id} {r.backbone.split('_')[0]}\n{r.GMAC:.2f} GMAC", (r.latency_b1_fp32_p50_ms, r.val_macro_f1), fontsize=8)
ax.set(xlabel="độ trễ batch 1, FP32, p50 (ms)", ylabel="macro-F1 val", title="Bước 1: chất lượng và độ trễ (cỡ điểm = #params)")
ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIG_DIR / "backbones_f1_latency.png", dpi=150); plt.show()

## Bước 2 — Công thức huấn luyện (≥ 3 trục)
Mỗi lần chạy chỉ khác `T00` **một yếu tố**. Mã `T01`, `T02`, ... Ghi rõ khác ở điểm nào.
Chạy trên backbone đã chọn ở Bước 1, 1 seed, 12 epoch. `T00` chạy lại đúng cấu hình của backbone đó ở Bước 1:
chênh lệch giữa hai lần (cùng seed) là thước đo nhiễu do GPU không tất định.

In [ ]:
AXES = {   # exp_id: (trục, khác T00 ở điểm nào, ghi đè Config)
    "T01": ("A", "init=frozen (chỉ train head)", dict(init="frozen")),
    "T02": ("A", "init=scratch (từ đầu)", dict(init="scratch")),
    "T03": ("B", "aug=TrivialAugmentWide", dict(aug="trivial")),
    "T04": ("B", "CutMix alpha=1", dict(mix="cutmix")),
    "T05": ("B", "thêm lật dọc", dict(aug="vflip")),
    "T06": ("C", "label smoothing 0.1", dict(loss="ls")),
    "T07": ("C", "focal loss gamma=2", dict(loss="focal")),
    "T08": ("C", "CE trọng số 1/n_c", dict(loss="ce_weighted")),
    "T09": ("D", "sampler cân bằng lớp", dict(sampler="balanced")),
    "T10": ("F", "EMA decay 0.999", dict(ema_decay=0.999)),
    "T11": ("E", "LR head = LR backbone = 1e-4", dict(lr_head=1e-4)),
}
DESC = {"T01": "A_frozen", "T02": "A_scratch", "T03": "B_trivialaug", "T04": "B_cutmix", "T05": "B_vflip",
        "T06": "C_labelsmooth", "T07": "C_focal", "T08": "C_weightedce", "T09": "D_balanced", "T10": "F_ema",
        "T11": "E_samelr"}
t_base = replace(b_cfgs[BB_EXP], exp_id="T00", desc="baseline")
t_cfgs = {"T00": t_base} | {e: replace(t_base, exp_id=e, desc=DESC[e], **ov) for e, (_, _, ov) in AXES.items()}
t_sum = dict(zip(t_cfgs, X.run_many(list(t_cfgs.values()), N_PARALLEL)))

In [ ]:
f1 = {e: s["val_macro_f1"] for e, s in t_sum.items()}
base_f1 = f1["T00"]
NONDET = abs(base_f1 - b_sum[BB_EXP]["val_macro_f1"])
MIN_GAIN = max(0.005, NONDET)
print(f"Nhiễu không tất định (T00 vs {BB_EXP}, cùng cấu hình + seed): |Δ| = {NONDET:.4f} → MIN_GAIN = {MIN_GAIN:.4f}")

winners = {}
for axis in sorted({a for a, _, _ in AXES.values()}):
    exps = [e for e, (a, _, _) in AXES.items() if a == axis]
    e_best = max(exps, key=f1.get)
    if f1[e_best] - base_f1 > MIN_GAIN:
        winners[axis] = e_best
print("Trục thắng:", {a: f"{e} ({AXES[e][1]}, Δ {f1[e] - base_f1:+.4f})" for a, e in winners.items()} or "không có")

if len(winners) >= 2:
    combo = {k: v for e in winners.values() for k, v in AXES[e][2].items()}
    AXES["T12"] = ("kết hợp", " + ".join(AXES[e][1] for e in winners.values()), combo)
    t_cfgs["T12"] = replace(t_base, exp_id="T12", desc="combo_" + "_".join(winners.values()), **combo)
    t_sum["T12"] = X.run_many([t_cfgs["T12"]], 1)[0]
    f1["T12"] = t_sum["T12"]["val_macro_f1"]

FINAL_EXP = max(t_sum, key=lambda e: (f1[e], -int(e[1:])))
final_cfg = t_cfgs[FINAL_EXP]
FINAL_REASON = f"{FINAL_EXP} ({'công thức nền' if FINAL_EXP == 'T00' else AXES[FINAL_EXP][1]}): macro-F1 val {f1[FINAL_EXP]:.4f} cao nhất trong các lần chạy T"
print("Cấu hình huấn luyện cuối:", FINAL_REASON)

tr = pd.DataFrame([{
    "exp_id": e, "backbone": s["backbone"], "axis": "nền" if e == "T00" else AXES[e][0],
    "change_vs_T00": "công thức nền" if e == "T00" else AXES[e][1], "seed": s["seed"],
    "val_macro_f1": s["val_macro_f1"], "val_top1": s["val_top1"], "delta_vs_T00": s["val_macro_f1"] - base_f1,
    "F1_ChineeApple": f1_of(s, "Chinee Apple"), "F1_SnakeWeed": f1_of(s, "Snake Weed"),
    "best_epoch": s["best_epoch"], "train_s_per_epoch": s["train_time_per_epoch_s"],
    "note": ("CHỌN (cấu hình cuối); " if e == FINAL_EXP else "") + ("thắng trục; " if e in winners.values() else "")
            + (f"EMA, F1 không EMA {s['val_macro_f1_raw_at_best']:.4f}" if "val_macro_f1_raw_at_best" in s else ""),
} for e, s in t_sum.items()])
display(tr.round(4))

fig, ax = plt.subplots(figsize=(9, 4.5))
d = tr[tr.exp_id != "T00"]
ax.bar(d.exp_id + "\n" + d.axis, d.delta_vs_T00, color=["tab:green" if v > MIN_GAIN else "tab:red" if v < -MIN_GAIN else "tab:gray" for v in d.delta_vs_T00])
ax.axhspan(-MIN_GAIN, MIN_GAIN, color="gray", alpha=0.15, label=f"±MIN_GAIN = {MIN_GAIN:.3f} (vùng nhiễu)")
ax.axhline(0, color="k", lw=0.8)
ax.set(ylabel="Δ macro-F1 val so với T00", title=f"Bước 2: ablation công thức huấn luyện ({BB}, T00 = {base_f1:.4f})")
ax.legend(); fig.tight_layout(); fig.savefig(FIG_DIR / "training_ablation.png", dpi=150); plt.show()

## Bước 3 — Suy luận (≥ 4 phương pháp) và độ trễ
Làm trên **val**. Không huấn luyện lại. Mã `I00` (1 view, mốc), `I01`, ...
Mô hình: lần chạy cấu hình cuối (seed 0). Một lượt qua val tính mọi view; mỗi phương pháp là một cách gộp view.
Dự đoán dùng AMP (FP16 autocast) nên độ trễ trong bảng `Inference` cũng đo ở AMP.

In [ ]:
fm, fcfg = X.load_model(run_dir(final_cfg), DEVICE)
MEAN, STD = fcfg["mean"], fcfg["std"]
VIEWS = X.tta_views(224)
val_full = X.full_loader(VAL_DF, IMAGES_DIR, MEAN, STD, batch_size=64, num_workers=NUM_WORKERS)
vnames, yv, VL = INF.predict_views(fm, val_full, DEVICE, VIEWS)
view_errors = VL.pop("_errors")
saved = np.load(run_dir(final_cfg) / "val_logits.npz")
assert list(saved["filenames"]) == vnames
print("Kiểm tra: view 'center' trùng tiền xử lý val lúc train, lệch logit lớn nhất =",
      float(np.abs(saved["logits"] - VL["center"]).max()))
print("View không áp dụng được:", view_errors or "không có")

crops = [f"crop{i}" for i in range(5)]
METHODS = {   # mã: (phương pháp, view, cách gộp)
    "I00": ("1 view (center crop 224)", ["center"], "prob"),
    "I01": ("TTA lật ngang", ["center", "center_hflip"], "prob"),
    "I02a": ("TTA 5 crop", crops, "prob"),
    "I02b": ("TTA 5 crop + lật (10 view)", crops + [c + "_hflip" for c in crops], "prob"),
    "I03a": ("TTA lật ngang, gộp LOGIT", ["center", "center_hflip"], "logit"),
    "I03b": ("TTA 10 view, gộp LOGIT", crops + [c + "_hflip" for c in crops], "logit"),
    "I04_224": ("ảnh nguyên resize về 224", ["full224"], "prob"),
    "I04_256": ("ảnh nguyên 256", ["full256"], "prob"),
    "I04_288": ("ảnh nguyên phóng 288", ["full288"], "prob"),
    "I04_320": ("ảnh nguyên phóng 320", ["full320"], "prob"),
}


def views_batch(keys):
    return lambda x: torch.cat([VIEWS[k](x) for k in keys])


inf_rows = []
for code_, (name, keys, space) in METHODS.items():
    if any(VL[k] is None for k in keys):
        inf_rows.append({"exp_id": code_, "method": name, "model": f"{FINAL_EXP} seed0", "K": len(keys),
                         "note": "không áp dụng được với backbone này: " + "; ".join(view_errors[k] for k in keys if k in view_errors)[:200]})
        continue
    probs = INF.aggregate_views([VL[k] for k in keys], space)
    m = compute_metrics(yv, probs.argmax(1), probs)
    lat = BM.tta_latency(fm, len(keys), views_batch(keys), 256, "amp", **LAT)
    inf_rows.append({"exp_id": code_, "method": name, "model": f"{FINAL_EXP} seed0", "K": len(keys), "agg": space,
                     "val_macro_f1": m["macro_f1"], "val_top1": m["top1"], "val_ece": m["ece"],
                     "p50_ms": lat["p50"], "p95_ms": lat["p95"], "p99_ms": lat["p99"], "images_per_s": lat["images_per_s"],
                     "F1_ChineeApple": m["f1"][0], "F1_SnakeWeed": m["f1"][7]})
inf = pd.DataFrame(inf_rows)

In [ ]:
# I05: ensemble 3 backbone tốt nhất của Bước 1 (1 view, công thức nền)
top3 = bb.sort_values("val_macro_f1", ascending=False).exp_id.head(3).tolist()
ens_logits = [np.load(run_dir(b_cfgs[e]) / "val_logits.npz") for e in top3]
assert all(list(z["filenames"]) == vnames for z in ens_logits)
p_ens = INF.ensemble_probs([INF.softmax(z["logits"]) for z in ens_logits])
m = compute_metrics(yv, p_ens.argmax(1), p_ens)
ens_models = [X.load_model(run_dir(b_cfgs[e]), DEVICE)[0] for e in top3]
x1 = torch.randn(1, 3, 224, 224, device=DEVICE).contiguous(memory_format=torch.channels_last)


def run_ens():
    with torch.inference_mode(), torch.autocast(DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == "cuda"):
        for mm in ens_models:
            mm(x1)


r = BM.bench(run_ens, LAT["warmup"], LAT["iters"], SYNC)
inf_rows.append({"exp_id": "I05", "method": "ensemble 3 backbone (trung bình xác suất)", "model": "+".join(top3),
                 "K": 3, "agg": "prob", "val_macro_f1": m["macro_f1"], "val_top1": m["top1"], "val_ece": m["ece"],
                 "p50_ms": r["p50"], "p95_ms": r["p95"], "p99_ms": r["p99"], "images_per_s": 1000 / r["p50"],
                 "F1_ChineeApple": m["f1"][0], "F1_SnakeWeed": m["f1"][7]})
del ens_models
torch.cuda.empty_cache()

# I06: trọng số EMA so với trọng số thường (cùng lần chạy T10, cùng epoch tốt nhất)
s10 = t_sum["T10"]
i00 = next(r for r in inf_rows if r["exp_id"] == "I00")
inf_rows.append({"exp_id": "I06", "method": "trọng số EMA vs không EMA (T10)", "model": "T10 seed0", "K": 1,
                 "val_macro_f1": s10["val_macro_f1"], "val_top1": s10["val_top1"], "val_ece": s10["val_ece"],
                 "p50_ms": i00["p50_ms"], "p95_ms": i00["p95_ms"], "p99_ms": i00["p99_ms"],
                 "note": f"không EMA cùng epoch: macro-F1 {s10['val_macro_f1_raw_at_best']:.4f}; EMA không tốn thêm lúc suy luận"})

# I07: temperature scaling trên 1 view (T khớp trên VAL)
T_val = INF.fit_temperature(VL["center"], yv)
p_ts = INF.apply_temperature(VL["center"], T_val)
m0, m1 = compute_metrics(yv, p_ts.argmax(1), INF.softmax(VL["center"])), compute_metrics(yv, p_ts.argmax(1), p_ts)
inf_rows.append({"exp_id": "I07", "method": f"1 view + temperature scaling (T = {T_val:.3f})", "model": f"{FINAL_EXP} seed0",
                 "K": 1, "val_macro_f1": m1["macro_f1"], "val_top1": m1["top1"], "val_ece": m1["ece"],
                 "p50_ms": i00["p50_ms"], "p95_ms": i00["p95_ms"], "p99_ms": i00["p99_ms"],
                 "note": f"ECE val {m0['ece']:.4f} → {m1['ece']:.4f}, NLL {m0['nll']:.4f} → {m1['nll']:.4f} (khớp và đo cùng trên val)"})

# I08: FP16 (model.half()) và gộp BatchNorm vào conv
center = {"c": VIEWS["center"]}
z32 = INF.predict_views(fm, val_full, DEVICE, center, amp=False)[2]["c"]
fm16 = copy.deepcopy(fm).half()
z16 = INF.predict_views(fm16, val_full, DEVICE, {"c": lambda x: VIEWS["center"](x).half()}, amp=False)[2]["c"]
variants = {"FP32": (fm, z32), "FP16 (model.half)": (fm16, z16)}
try:
    fused = INF.fuse_conv_bn(fm)
    N_FUSED = fused.n_fused
except Exception as e:   # noqa: BLE001
    fused, N_FUSED = None, f"lỗi: {e}"
if fused is not None and N_FUSED:
    zf = INF.predict_views(fused, val_full, DEVICE, center, amp=False)[2]["c"]
    variants["FP32 + gộp BN"] = (fused, zf)
for name, (mm, z) in variants.items():
    p = INF.softmax(z)
    mt = compute_metrics(yv, p.argmax(1), p)
    dt = "fp16" if "FP16" in name else "fp32"
    lat = BM.latency_report(mm, 1, 224, dt, fused_bn="BN" in name, **LAT)
    inf_rows.append({"exp_id": "I08", "method": name, "model": f"{FINAL_EXP} seed0", "K": 1,
                     "val_macro_f1": mt["macro_f1"], "val_top1": mt["top1"], "val_ece": mt["ece"],
                     "p50_ms": lat["p50"], "p95_ms": lat["p95"], "p99_ms": lat["p99"], "images_per_s": lat["images_per_s"],
                     "note": f"lệch logit lớn nhất so với FP32: {np.abs(z - z32).max():.2e}"})
if not N_FUSED:
    inf_rows.append({"exp_id": "I08", "method": "gộp BN", "model": BB, "note": f"không áp dụng: backbone không có BatchNorm ({N_FUSED})"})

inf = pd.DataFrame(inf_rows)
base_p50 = inf.loc[inf.exp_id == "I00", "p50_ms"].iloc[0]
inf["relative_cost_vs_I00"] = inf["p50_ms"] / base_p50
display(inf.round(4))

In [ ]:
# Độ trễ chi tiết của mô hình cuối: batch 1 và 32, FP32 / AMP / FP16, có/không gộp BN
for bs in (1, 32):
    for name, mm in [("thường", fm)] + ([("gộp BN", fused)] if fused is not None and N_FUSED else []):
        for dt in ("fp32", "amp", "fp16"):
            lat_rows.append({"config": f"{FINAL_EXP} {BB} ({name})", "step": "I",
                             **BM.latency_report(mm, bs, 224, dt, fused_bn=name == "gộp BN", **LAT)})
lat_df = pd.DataFrame(lat_rows)
display(lat_df[lat_df.step == "I"].round(3))

# Chọn phương pháp suy luận cuối (quy tắc đặt trước): 1 mô hình, p95 batch-1 ≤ ngân sách, macro-F1 val cao nhất
cand = inf[inf.exp_id.isin(list(METHODS)) & inf.val_macro_f1.notna() & (inf.p95_ms <= LATENCY_BUDGET_MS)]
best = cand.sort_values(["val_macro_f1", "p50_ms"], ascending=[False, True]).iloc[0]
FINAL_METHOD = best.exp_id
FINAL_KEYS, FINAL_SPACE = METHODS[FINAL_METHOD][1], METHODS[FINAL_METHOD][2]
FINAL_P95 = float(best.p95_ms)
print(f"Phương pháp suy luận cuối: {FINAL_METHOD} ({best.method}), macro-F1 val {best.val_macro_f1:.4f}, "
      f"p95 batch-1 {FINAL_P95:.2f} ms + temperature scaling")

fig, ax = plt.subplots(figsize=(8, 5))
d = inf.dropna(subset=["val_macro_f1", "p50_ms"])
ax.scatter(d.p50_ms, d.val_macro_f1)
for _, r in d.iterrows():
    ax.annotate(f"{r.exp_id} {r.method[:22]}", (r.p50_ms, r.val_macro_f1), fontsize=7)
ax.set(xscale="log", xlabel="độ trễ batch 1, p50 (ms, log)", ylabel="macro-F1 val",
       title=f"Bước 3: đánh đổi độ chính xác và độ trễ ({FINAL_EXP} {BB})")
ax.grid(alpha=0.3); fig.tight_layout(); fig.savefig(FIG_DIR / "inference_tradeoff.png", dpi=150); plt.show()
del fm16
torch.cuda.empty_cache()

## Bước 4 — Chung kết: ≥ 3 seed, test một lần mỗi seed
1. Chốt cấu hình trên **val**.
2. Chạy cấu hình cuối **và mốc** (`T00` + `I00`) với cùng số seed, bật `save_test_predictions=True`.
3. Dự đoán lưu ở `predictions/<exp_id>_seed<k>_test.csv`. Rồi chạy `eval.py` (ô dưới).
Seed 0 của `F01` chính là lần chạy cấu hình cuối ở Bước 2 (cùng cấu hình, cùng seed, chỉ sao chép); seed 0 của mốc là `T00`.
Test chạy **đúng một lần** cho mỗi (cấu hình, seed) trong ô dưới.

In [ ]:
SEEDS = (0, 1, 2)


def same_cfg(a, b):
    strip = lambda c: {k: v for k, v in asdict(c).items() if k not in ("exp_id", "desc")}
    return strip(a) == strip(b)


base_cfgs = {k: replace(t_base, seed=k) for k in SEEDS}
f_cfgs = {k: replace(final_cfg, exp_id="F01", desc=f"final_{FINAL_EXP}", seed=k) for k in SEEDS}
X.alias_run(final_cfg, f_cfgs[0])
X.run_many([base_cfgs[1], base_cfgs[2]] + [f_cfgs[k] for k in (1, 2) if not same_cfg(f_cfgs[k], base_cfgs[k])], N_PARALLEL)
for k in (1, 2):
    if same_cfg(f_cfgs[k], base_cfgs[k]):
        X.alias_run(base_cfgs[k], f_cfgs[k])

T_by_seed = {}
for k in SEEDS:
    # Chung kết F01: phương pháp suy luận đã chốt + temperature scaling khớp trên VAL
    m, c = X.load_model(run_dir(f_cfgs[k]), DEVICE)
    vk = {kk: VIEWS[kk] for kk in FINAL_KEYS}
    vn, vy, vz = INF.predict_views(m, X.full_loader(VAL_DF, IMAGES_DIR, c["mean"], c["std"], 64, NUM_WORKERS), DEVICE, vk)
    tn, ty, tz = INF.predict_views(m, X.full_loader(TEST_DF, IMAGES_DIR, c["mean"], c["std"], 64, NUM_WORKERS), DEVICE, vk)
    vz, tz = [vz[kk] for kk in FINAL_KEYS], [tz[kk] for kk in FINAL_KEYS]
    T_by_seed[k] = X.fit_temperature_views(vz, vy, FINAL_SPACE)
    save_predictions(PRED_DIR / f"F01_seed{k}_val.csv", vn, vy, X.apply_temperature_views(vz, T_by_seed[k], FINAL_SPACE))
    save_predictions(PRED_DIR / f"F01_seed{k}_test.csv", tn, ty, X.apply_temperature_views(tz, T_by_seed[k], FINAL_SPACE))
    save_predictions(PRED_DIR / f"F01_uncal_seed{k}_test.csv", tn, ty, X.apply_temperature_views(tz, 1.0, FINAL_SPACE))
    # Mốc T00 + I00: 1 view, không temperature scaling
    mb, cb = X.load_model(run_dir(base_cfgs[k]), DEVICE)
    bn_, by, bz = INF.predict_views(mb, X.full_loader(VAL_DF, IMAGES_DIR, cb["mean"], cb["std"], 64, NUM_WORKERS), DEVICE, {"c": VIEWS["center"]})
    save_predictions(PRED_DIR / f"T00_seed{k}_val.csv", bn_, by, INF.softmax(bz["c"]))
    bn_, by, bz = INF.predict_views(mb, X.full_loader(TEST_DF, IMAGES_DIR, cb["mean"], cb["std"], 64, NUM_WORKERS), DEVICE, {"c": VIEWS["center"]})
    save_predictions(PRED_DIR / f"T00_seed{k}_test.csv", bn_, by, INF.softmax(bz["c"]))
    del m, mb
    torch.cuda.empty_cache()
    print(f"seed {k}: T = {T_by_seed[k]:.3f}, đã ghi predictions F01/T00")
(OUT_DIR / "final_choice.json").write_text(json.dumps({
    "backbone": BB, "backbone_reason": BB_REASON, "final_training": FINAL_EXP, "final_training_reason": FINAL_REASON,
    "final_inference": FINAL_METHOD, "final_views": FINAL_KEYS, "final_agg": FINAL_SPACE, "final_p95_ms_b1_amp": FINAL_P95,
    "temperature_by_seed": T_by_seed, "seeds": list(SEEDS), "min_gain": MIN_GAIN, "nondeterminism": NONDET,
    "config": asdict(final_cfg)}, indent=2, ensure_ascii=False))
FINAL_P95_STR = f"{FINAL_P95:.2f}"
EVAL_OUT = OUT_DIR / "eval_out"

In [ ]:
# Tính chỉ số theo đúng định nghĩa của lớp (đã viết sẵn, không sửa eval.py):
!python {REPO_DIR}/eval.py score --pred "{PRED_DIR}/F01_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01 --out {EVAL_OUT}
!python {REPO_DIR}/eval.py score --pred "{PRED_DIR}/F01_uncal_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag F01_uncal --out {EVAL_OUT}
!python {REPO_DIR}/eval.py score --pred "{PRED_DIR}/T00_seed*_test.csv" \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --tag T00 --out {EVAL_OUT}

In [ ]:
# Tự chấm phần I của RUBRIC (đề xuất, giảng viên xác nhận)
!python {REPO_DIR}/eval.py grade --final "{PRED_DIR}/F01_seed*_test.csv" \
    --baseline "{PRED_DIR}/T00_seed*_test.csv" --uncal "{PRED_DIR}/F01_uncal_seed*_test.csv" \
    --final-val "{PRED_DIR}/F01_seed*_val.csv" --val-csv {LABELS_DIR}/val_subset0.csv \
    --latency-p95-ms {FINAL_P95_STR} --latency-method proper \
    --test-csv {LABELS_DIR}/test_subset0.csv --labels {LABELS_DIR}/labels.csv --out {EVAL_OUT}

In [ ]:
# Ma trận nhầm lẫn (tổng 3 seed, chuẩn hoá theo hàng) và phân tích lỗi
cm = pd.read_csv(EVAL_OUT / "F01_confusion_sum.csv", index_col=0).to_numpy()
cmn = cm / cm.sum(1, keepdims=True)
fig, ax = plt.subplots(figsize=(8, 6.5))
im = ax.imshow(cmn, cmap="Blues", vmin=0, vmax=1)
for i in range(9):
    for j in range(9):
        ax.text(j, i, f"{cmn[i, j]:.2f}\n({cm[i, j]})", ha="center", va="center", fontsize=6.5,
                color="white" if cmn[i, j] > 0.5 else "black")
ax.set_xticks(range(9), D.CLASS_NAMES, rotation=40, ha="right"); ax.set_yticks(range(9), D.CLASS_NAMES)
ax.set(xlabel="dự đoán", ylabel="nhãn thật", title="F01 trên test: ma trận nhầm lẫn (tổng 3 seed, tỉ lệ theo hàng)")
fig.colorbar(im); fig.tight_layout(); fig.savefig(FIG_DIR / "final_confusion_test.png", dpi=150); plt.show()

off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted(((off[i, j], D.CLASS_NAMES[i], D.CLASS_NAMES[j]) for i in range(9) for j in range(9) if off[i, j]), reverse=True)[:5]
print("Cặp nhầm nhiều nhất (thật → đoán, tổng 3 seed):", [(f"{a} → {b}", int(n)) for n, a, b in pairs])

p0 = read_pred(str(PRED_DIR / "F01_seed0_test.csv"))
wrong = np.flatnonzero(p0.y_true != p0.y_pred)
pick = wrong[np.argsort(-p0.probs[wrong].max(1))][:12]
if len(pick):
    fig, axes = plt.subplots(2, 6, figsize=(15, 5.6))
    for ax, i in zip(axes.flat, pick):
        ax.imshow(Image.open(IMAGES_DIR / p0.filenames[i]))
        ax.set_title(f"thật: {D.CLASS_NAMES[p0.y_true[i]]}\nđoán: {D.CLASS_NAMES[p0.y_pred[i]]} ({p0.probs[i].max():.2f})", fontsize=8)
    for ax in axes.flat:
        ax.axis("off")
    fig.suptitle("F01 seed 0: ảnh test bị đoán sai với độ tin cậy cao nhất")
    fig.tight_layout(); fig.savefig(FIG_DIR / "final_errors_test.jpg", dpi=100, pil_kwargs={"quality": 85}); plt.show()

## Bước 5 — Sản phẩm
`results.xlsx` (các sheet ở GUIDE.md mục 6.1), `curves/<exp_id>_*.png`, `report.md`, `predictions/`, `code/`.
Ô dưới ghi `results.xlsx` (7 sheet) và `report_draft.md` (bản nháp đã điền số liệu; phần phân tích bạn tự viết vào `report.md`).

In [ ]:
def fmt_ms(mean, std):
    return f"{mean:.4f} ± {std:.4f}"


per_seed = {t: pd.read_csv(EVAL_OUT / f"{t}_per_seed.csv") for t in ("F01", "F01_uncal", "T00")}
final_rows = []
for tag, exp_desc in [("F01", f"{BB} + {FINAL_EXP} + {FINAL_METHOD} + TS"), ("T00", f"{BB} + T00 + I00 (mốc)")]:
    ps = per_seed[tag]
    for _, r in ps.iterrows():
        vp = read_pred(str(PRED_DIR / f"{tag}_seed{int(r.seed)}_val.csv"))
        vm = compute_metrics(vp.y_true, vp.y_pred, vp.probs)
        final_rows.append({"exp_id": tag, "config": exp_desc, "seed": int(r.seed), "val_macro_f1": vm["macro_f1"],
                           "test_macro_f1": r.macro_f1, "test_top1": r.top1, "test_balanced_acc": r.balanced_acc, "test_ece": r.ece})
    sub = pd.DataFrame([x for x in final_rows if x["exp_id"] == tag])
    final_rows.append({"exp_id": tag, "config": exp_desc, "seed": f"mean ± std ({len(sub)} seed, ddof=1)",
                       **{c: fmt_ms(sub[c].mean(), sub[c].std(ddof=1)) for c in ("val_macro_f1", "test_macro_f1", "test_top1", "test_balanced_acc", "test_ece")}})
ece_u = per_seed["F01_uncal"]["ece"]
final_rows.append({"exp_id": "F01_uncal", "config": "F01 chưa temperature scaling", "seed": "mean ± std",
                   "test_ece": fmt_ms(ece_u.mean(), ece_u.std(ddof=1))})
final_df = pd.DataFrame(final_rows)

pc_rows = []
for tag in ("F01", "T00"):
    pc = pd.read_csv(EVAL_OUT / f"{tag}_per_class.csv")
    for _, r in pc.iterrows():
        pc_rows.append({"config": tag, "class": r["class"], "n_test": int(r.support),
                        "precision": fmt_ms(r.precision_mean, r.precision_std), "recall": fmt_ms(r.recall_mean, r.recall_std),
                        "f1": fmt_ms(r.f1_mean, r.f1_std)})
perclass_df = pd.DataFrame(pc_rows)

summ = pd.concat([
    bb.assign(type="backbone", description=bb.backbone, latency_ms=bb.latency_b1_fp32_p50_ms)[["exp_id", "type", "description", "val_macro_f1", "val_top1", "latency_ms"]],
    tr.assign(type="training", description=tr.change_vs_T00, latency_ms=np.nan)[["exp_id", "type", "description", "val_macro_f1", "val_top1", "latency_ms"]],
    inf.dropna(subset=["val_macro_f1"]).assign(type="inference", description=inf.method, latency_ms=inf.p50_ms)[["exp_id", "type", "description", "val_macro_f1", "val_top1", "latency_ms"]],
]).sort_values("val_macro_f1", ascending=False).head(10)
summ["latency_ms"] = summ["latency_ms"].round(2)
f_test = per_seed["F01"].macro_f1; b_test = per_seed["T00"].macro_f1
headline = pd.DataFrame([
    {"item": "Cấu hình cuối", "value": f"{BB} + {FINAL_EXP} ({FINAL_REASON}) + {FINAL_METHOD} + temperature scaling"},
    {"item": "macro-F1 test F01", "value": fmt_ms(f_test.mean(), f_test.std(ddof=1))},
    {"item": "macro-F1 test mốc T00+I00", "value": fmt_ms(b_test.mean(), b_test.std(ddof=1))},
    {"item": "Δ macro-F1 test", "value": f"{f_test.mean() - b_test.mean():+.4f}"},
    {"item": "top-1 test F01", "value": fmt_ms(per_seed['F01'].top1.mean(), per_seed['F01'].top1.std(ddof=1))},
    {"item": "p95 batch-1 (AMP) của phương pháp cuối", "value": f"{FINAL_P95:.2f} ms"},
])

SHEETS = {"Backbones": bb, "Training": tr, "Inference": inf, "Final": final_df, "PerClass": perclass_df,
          "Latency": lat_df.drop(columns=["n"], errors="ignore"), "Summary": summ}
xlsx = OUT_DIR / "results.xlsx"
with pd.ExcelWriter(xlsx, engine="openpyxl") as w:
    for name, df in SHEETS.items():
        df.to_excel(w, sheet_name=name, index=False)
    headline.to_excel(w, sheet_name="Summary", index=False, startrow=len(summ) + 3)
from openpyxl.styles import Font, PatternFill
from openpyxl import load_workbook
wb = load_workbook(xlsx)
for ws in wb.worksheets:
    ws.freeze_panes = "A2"
    head = [c.value for c in ws[1]]
    for col in ws.columns:
        ws.column_dimensions[col[0].column_letter].width = min(48, max(10, *(len(str(c.value or "")) for c in col[:40])) + 2)
        for c in col[1:]:
            if isinstance(c.value, float):
                c.number_format = "0.0000"
    for c in ws[1]:
        c.font = Font(bold=True)
    if "val_macro_f1" in head and ws.title in ("Backbones", "Training", "Inference", "Summary"):
        j = head.index("val_macro_f1") + 1
        vals = [(ws.cell(i, j).value, i) for i in range(2, ws.max_row + 1) if isinstance(ws.cell(i, j).value, float)]
        if vals:
            row = max(vals)[1]
            for c in ws[row]:
                c.fill = PatternFill("solid", fgColor="FFF2CC")
                c.font = Font(bold=True)
wb.save(xlsx)
print("Đã ghi", xlsx, "| sheet:", wb.sheetnames)

missing = [s["exp_id"] for s in [*b_sum.values(), *t_sum.values()] if not Path(s["curve"]).exists()]
print("curves:", len(list(CURVES_DIR.glob("*.png"))), "ảnh | thiếu:", missing or "không")

In [ ]:
def md_table(df, digits=4):
    df = df.copy()
    for c in df.columns:
        if df[c].dtype.kind == "f":
            df[c] = df[c].map(lambda v: "" if pd.isna(v) else f"{v:.{digits}f}")
    rows = ["| " + " | ".join(map(str, df.columns)) + " |", "|" + "---|" * len(df.columns)]
    rows += ["| " + " | ".join("" if pd.isna(v) else str(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(rows)


grade = json.loads((EVAL_OUT / "grade_I.json").read_text())
f01 = json.loads((EVAL_OUT / "F01_summary.json").read_text())
names = f01["classes"]
hard = {c: f01["recall"]["mean"][i] for i, c in enumerate(names) if c.lower().replace(" ", "") in ("chineeapple", "snakeweed")}
report = f"""# Báo cáo Lab Day 2 — DeepWeeds (BẢN NHÁP TỰ SINH)

> Sinh tự động từ lần chạy {ENV['platform']} ({ENV['gpu']}, torch {ENV['torch']}, timm {ENV['timm']}, commit {ENV['git_commit']}).
> Mọi số dưới đây lấy từ `results.xlsx` / `eval_out/`. Các mục **TODO** là phần phân tích bạn tự viết.

## 1. Tóm tắt
- Cấu hình cuối: **{BB}** + công thức **{FINAL_EXP}** + suy luận **{FINAL_METHOD}** + temperature scaling.
- macro-F1 test (3 seed): **{fmt_ms(f_test.mean(), f_test.std(ddof=1))}** so với mốc T00+I00 {fmt_ms(b_test.mean(), b_test.std(ddof=1))}
  (Δ {f_test.mean() - b_test.mean():+.4f}); top-1 test {fmt_ms(per_seed['F01'].top1.mean(), per_seed['F01'].top1.std(ddof=1))}.
- Recall lớp khó (test, trung bình 3 seed): {', '.join(f'{c} {v * 100:.1f}%' for c, v in hard.items())}.
- TODO: kết luận chính.

## 2. Dữ liệu và thiết lập
- DeepWeeds fold 0: {split_stats['n']} ảnh (train/val/test/tổng); giao từng cặp rỗng; hợp {split_stats['union']}; không thiếu file.
  Một ảnh lệch nhãn giữa `train_subset0.csv` và `labels.csv` (xem `figures/split_checks.json`), dùng nguyên nhãn fold theo S1.
- Phân bố lớp: `figures/eda_class_distribution.png`; mất cân bằng train {c_tr.max() / c_tr.min():.2f}x (Negatives / lớp nhỏ nhất).
- Kiểm tra pipeline (`figures/pipeline_checks.json`): loss ban đầu ≈ ln 9 cho mọi backbone sau khi khởi tạo lại head;
  overfit 1 batch tới loss {overfit_hist[-1]:.4f}; BN eval/train được kiểm tra.
- Công thức nền T00: AdamW, LR backbone 1e-4 / head 1e-3, wd 0,05 (không cho norm/bias), warmup 1 epoch + cosine,
  CE, batch 64, 12 epoch, AMP, RandomResizedCrop(224) + lật ngang; val/test: Resize 256 + CenterCrop 224.
- Chọn checkpoint: epoch có macro-F1 val cao nhất. Seed: 0 cho Bước 1-3; 0, 1, 2 cho chung kết.

## 3. So sánh backbone
{md_table(bb[['exp_id', 'backbone', 'weight_tag', 'params_M', 'GMAC', 'val_macro_f1', 'val_top1', 'train_s_per_epoch', 'latency_b1_fp32_p50_ms', 'note']])}

Chọn: {BB_REASON}

TODO: nhận xét (hội tụ, quá khớp, thứ hạng so với ImageNet, FLOPs vs độ trễ). Hình: `figures/backbones_f1_latency.png`.

## 4. Công thức huấn luyện ({BB}, 1 seed)
Nhiễu không tất định (T00 vs {BB_EXP}, cùng cấu hình + seed): {NONDET:.4f}; ngưỡng thắng MIN_GAIN = {MIN_GAIN:.4f}. Cách làm: tham lam theo trục.

{md_table(tr[['exp_id', 'axis', 'change_vs_T00', 'val_macro_f1', 'val_top1', 'delta_vs_T00', 'F1_ChineeApple', 'F1_SnakeWeed', 'note']])}

Cấu hình cuối: {FINAL_REASON}. TODO: yếu tố nào giúp/không giúp và vì sao. Hình: `figures/training_ablation.png`.

## 5. Suy luận ({FINAL_EXP} seed 0, val)
{md_table(inf[['exp_id', 'method', 'K', 'val_macro_f1', 'val_top1', 'val_ece', 'p50_ms', 'p95_ms', 'relative_cost_vs_I00', 'note']])}

Phương pháp cuối: {FINAL_METHOD} (p95 batch-1 {FINAL_P95:.2f} ms). TODO: đánh đổi ngoại tuyến vs thời gian thực. Hình: `figures/inference_tradeoff.png`.

## 6. Chung kết (test, mỗi seed chạy một lần)
{md_table(final_df)}

Tự chấm phần I (`eval.py grade`): {grade['total']} / {grade['max_scored']}.
{md_table(pd.DataFrame(grade['items'])[['code', 'criterion', 'points', 'max', 'note']])}

Ma trận nhầm lẫn: `figures/final_confusion_test.png`; ảnh sai: `figures/final_errors_test.jpg`.
TODO: lớp nào còn nhầm nhiều nhất và giả thuyết nguyên nhân.

## 7. Kết luận và khuyến nghị
TODO: cấu hình tốt nhất, yếu tố đóng góp nhiều nhất, lựa chọn cho robot 30–100 ms/khung.

## 8. Hạn chế
- Một fold, chia ngẫu nhiên (không theo địa điểm); ablation 1 seed; 12 epoch (bài báo ~100 epoch).
- TODO.

## 9. Phụ lục
Cấu hình đầy đủ: `runs/<exp_id>/seed<k>/config.json`; lựa chọn: `final_choice.json`; môi trường: `env.json`.
"""
(OUT_DIR / "report_draft.md").write_text(report)
print("Đã ghi", OUT_DIR / "report_draft.md")
print("\n".join(report.splitlines()[:14]))